In [ ]:
import pandas as pd 
import geopandas as gpd 

df=pd.read_excel("NTAD_Military_Bases_-4186673718369528560.xlsx") 

df

,OBJECTID,Country,Feature Description,Feature Name,Controlled Unclassified Information Indicator,Is FIRRMA Site,Is Joint Base,Media Identifier,Primary Key Identifier,Globally Unique Identifier,Site Name,Site Operational Status,Site Reporting Component Code,State Name Code,Shape__Area,Shape__Length
0,1,usa,na,Devens Reserve Forces Tng Area,no,no,no,na,,f9f35d60-b757-448a-9683-ebc2e518a310,Devens Reserve Forces Tng Area,act,usar,ma,0.002253,0.324854
1,2,usa,na,Fort Campbell,no,yes,no,na,,7dc7f20f-6983-4e95-a593-a3299fb37d9f,Fort Campbell,act,usa,tn,0.042675,1.388270
2,3,usa,na,NG Snake Creek TS Miramar,no,no,no,na,,e33df005-0713-4e64-8ebe-10dd700cbd60,NG Snake Creek TS Miramar,act,armyNationalGuard,fl,0.000116,0.046478
3,4,usa,na,Piñon Canyon Maneuver Site,no,yes,no,na,,0582f20a-d60a-4468-89e0-6fdade6f3e0c,Piñon Canyon Maneuver Site,act,usa,co,0.097109,1.776971
4,5,usa,na,Stewart Annex,no,no,no,na,,0d629ab9-bead-4263-8d09-663887ea1100,Stewart Annex,act,usa,ny,0.000004,0.009891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819,983,usa,na,Sioux Gateway Airport ANG-CE Site 2,no,no,no,na,,NaN,Sioux Gateway Airport ANG-CE Site 2,act,airNationalGuard,ia,0.000007,0.018293
820,985,usa,na,Tonopah AFS Z164,no,no,no,na,,NaN,Tonopah AFS Z164,act,usaf,nv,0.000021,0.028547
821,989,usa,na,NUWC Fishers Island NY,no,no,no,na,,c798468f-742f-4426-8978-772ef856ea34,NUWC Fishers Island NY,act,usn,ny,0.000029,0.025207
822,991,usa,na,Grand Forks AFB,no,yes,no,na,,c9fabeb6-6bdf-4b89-ba4a-cb22a3b68cde,Grand Forks AFB,act,usaf,nd,0.002721,0.343547


In [39]:
#Preprocessing and Cleaning Dataset 
df_cleaned = df.drop(columns = ['OBJECTID','Country','Feature Description','Feature Name','Media Identifier','Primary Key Identifier','Globally Unique Identifier']) 



df_cleaned

,Controlled Unclassified Information Indicator,Is FIRRMA Site,Is Joint Base,Site Name,Site Operational Status,Site Reporting Component Code,State Name Code,Shape__Area,Shape__Length
0,no,no,no,Devens Reserve Forces Tng Area,act,usar,ma,0.002253,0.324854
1,no,yes,no,Fort Campbell,act,usa,tn,0.042675,1.388270
2,no,no,no,NG Snake Creek TS Miramar,act,armyNationalGuard,fl,0.000116,0.046478
3,no,yes,no,Piñon Canyon Maneuver Site,act,usa,co,0.097109,1.776971
4,no,no,no,Stewart Annex,act,usa,ny,0.000004,0.009891
...,...,...,...,...,...,...,...,...,...
819,no,no,no,Sioux Gateway Airport ANG-CE Site 2,act,airNationalGuard,ia,0.000007,0.018293
820,no,no,no,Tonopah AFS Z164,act,usaf,nv,0.000021,0.028547
821,no,no,no,NUWC Fishers Island NY,act,usn,ny,0.000029,0.025207
822,no,yes,no,Grand Forks AFB,act,usaf,nd,0.002721,0.343547


In [ ]:
import folium #For Interactive Map Visualization!

#Generate a GeoDataFrame from cleaned DataFrame
gdf = gpd.GeoDataFrame(df_cleaned)

In [41]:
#Implementation of Choropleth Map Visualization for Military Bases Distribution by State 
import json 
import requests #Necessary for Fetching GeoJSON Data for US States


#Renaming State Name Code 
gdf["state_abbreviation"]=gdf["State Name Code"].str.strip().str.upper() 

#Grouping States 
state_counts = gdf.groupby("state_abbreviation").size().reset_index(name="base_count") 

#Fetching GeoJSON Data for US States 
state_geo = requests.get("https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json").json() 


map_center = [39.50, -98.35] #Center of United States for Map Visualization 
military_bases_map = folium.Map(location=map_center, zoom_start=4) #Initialization of Folium Map Visualization

In [42]:
#Choropleth Map Visualization for Military Bases Distribution by State 
folium.Choropleth( 
    geo_data=state_geo, 
    name="Military Base Distribution (By US State)", 
    data=state_counts, 
    columns=["state_abbreviation", "base_count"], 
    key_on="feature.id", 
    fill_color="YlOrRd", 
    fill_opacity=0.7, 
    line_opacity=0.2, 
    legend_name="State Distribution of Military Bases", 
    highlight=True, 
).add_to(military_bases_map) 

#State Center Coordinates Mapping for Visualization
state_center_coordinates = { 
    "AL":[32.78,-86.83],"AK":[64.20,-153.49],"AZ":[34.27,-111.66],"AR":[34.89,-92.44],
    "CA":[37.18,-119.47],"CO":[38.99,-105.55],"CT":[41.62,-72.73],"DE":[38.99,-75.51],
    "FL":[28.63,-82.45],"GA":[32.64,-83.44],"HI":[20.29,-156.37],"ID":[44.35,-114.61],
    "IL":[40.04,-89.20],"IN":[39.89,-86.28],"IA":[42.07,-93.50],"KS":[38.49,-98.38],
    "KY":[37.53,-85.30],"LA":[31.07,-91.99],"ME":[45.37,-69.24],"MD":[39.05,-76.79],
    "MA":[42.26,-71.81],"MI":[44.35,-85.41],"MN":[46.28,-94.31],"MS":[32.74,-89.67],
    "MO":[38.36,-92.46],"MT":[47.05,-109.63],"NE":[41.54,-99.80],"NV":[39.33,-116.63],
    "NH":[43.68,-71.58],"NJ":[40.19,-74.67],"NM":[34.41,-106.11],"NY":[42.95,-75.53],
    "NC":[35.56,-79.39],"ND":[47.45,-100.47],"OH":[40.29,-82.79],"OK":[35.59,-97.49],
    "OR":[43.93,-120.56],"PA":[40.88,-77.80],"RI":[41.68,-71.56],"SC":[33.92,-80.90],
    "SD":[44.44,-100.23],"TN":[35.86,-86.35],"TX":[31.48,-99.33],"UT":[39.32,-111.09],
    "VT":[44.07,-72.67],"VA":[37.52,-78.85],"WA":[47.38,-120.45],"WV":[38.64,-80.62],
    "WI":[44.62,-89.99],"WY":[42.99,-107.55],"DC":[38.91,-77.02] 
    } 


In [43]:
#Branch Mapping 
branches= { 
    "usa": "Army", "usar": "Army Reserve", "armyNationalGuard": "Army NG",
    "usaf": "Air Force", "afr": "Air Force Reserve", "airNationalGuard": "Air NG",
    "usn": "Navy", "usnr": "Navy Reserve",
    "usmc": "Marines", "usmcr": "Marine Reserve",
    "whs": "WHS/Pentagon", "other": "Other"   
} 

#Number of Branches by State 
branch_counts = gdf.groupby(["state_abbreviation", "Site Reporting Component Code"]).size().reset_index(name="base_count") 



#Iterating through states 

for _, row in state_counts.iterrows(): 
    ab = row["state_abbreviation"] 
    if ab not in state_center_coordinates: 
        continue
    lat, lon = state_center_coordinates[ab]


    branch_lines = "" 

    for _, row in branch_counts.sort_values("base_count", ascending=False).iterrows(): 
        label = branches.get(row["Site Reporting Component Code"],
                                row["Site Reporting Component Code"])
        branch_lines += f"<tr><td>{label}</td><td style='text-align:right'>{row['base_count']}</td></tr>"

        popup_html = f"""
        <div style="font-family:Arial;width:200px;font-size:13px;">
        <h4 style="margin:0 0 5px;border-bottom:2px solid #333;">{ab}</h4>
        <b>Total Bases: {row['base_count']}</b>
        <table style="width:100%;margin-top:4px;">
            <tr style="background:#f0f0f0;font-weight:bold;">
            <td>Branch</td><td style="text-align:right">Count</td>
            </tr>
            {branch_lines}
        </table>
        </div>"""

    folium.LayerControl().add_to(military_bases_map)

In [44]:
military_bases_map.save("military_bases_map.html") #Saving the interactive map as an HTML file for sharing